In [ ]:
X = data[features]
y = data["Hopkins_category"].map({"YES": 1, "NO": 0})

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression()
lr.fit(X_train_scaled, y_train)

y_pred_lr = lr.predict(X_test_scaled)

In [ ]:
from sklearn.svm import SVC

svm = SVC(kernel="rbf", probability=True)
svm.fit(X_train_scaled, y_train)

y_pred_svm = svm.predict(X_test_scaled)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)  

y_pred_rf = rf.predict(X_test)

In [ ]:
from xgboost  import XGBClassifier

rf = XGBClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)  

y_pred_rf = rf.predict(X_test)

In [ ]:
from sklearn.metrics import classification_report, accuracy_score

def evaluate(y_true, y_pred, name):
    print(f"\n{name}")
    print("Accuracy:", accuracy_score(y_true, y_pred))
    print(classification_report(y_true, y_pred))

evaluate(y_test, y_pred_lr, "Logistic Regression")
evaluate(y_test, y_pred_svm, "SVM")
evaluate(y_test, y_pred_rf, "Random Forest")
evaluate(y_test, y_pred_rf, "XGBClassifier")

In [ ]:
model = LSTMModel(input_size=X.shape[2])
pos_weight = torch.tensor([len(y_train) / y_train.sum()])

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)


for epoch in range(20):

    model.train()

    outputs = model(X_train).squeeze()

    if torch.isnan(outputs).any():
        print("NaN in outputs!")
        break

    loss = criterion(outputs, y_train)

    if torch.isnan(loss):
        print("NaN in loss!")
        break

    optimizer.zero_grad()
    loss.backward()

    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

    optimizer.step()

    print(f"Epoch {epoch}, Loss: {loss.item():.4f}")

In [ ]:
import torch.nn as nn

class GRUModel(nn.Module):
    def __init__(self, input_size, hidden_size=32):
        super().__init__()

        self.gru = nn.GRU(input_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.gru(x)
        out = out[:, -1, :]
        out = self.fc(out)
        return out

model = GRUModel(input_size=X.shape[2])

In [ ]:
for epoch in range(20):

    model.train()

    outputs = model(X_train).squeeze()

    # check NaN
    if torch.isnan(outputs).any():
        print("NaN in outputs!")
        break

    loss = criterion(outputs, y_train)

    if torch.isnan(loss):
        print("NaN in loss!")
        break

    optimizer.zero_grad()
    loss.backward()

    # gradient clipping 
    torch.nn.utils.clip_grad_norm_(model.parameters(), 0.5)

    optimizer.step()

    print(f"Epoch {epoch}, Loss: {loss.item():.4f}")